<a href="https://colab.research.google.com/github/NatashaO15/intro-ciencia-datos/blob/main/Trabajo%20final/Proyecto_supply_chain_final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## PROYECTO DE CIENCIA DE DATOS: CADENA DE SUMINISTRO
#Natasha Orozco Zapata
#Andrés Zabala Restrepo


**LIMPIEZA DEL DATASET**

In [1]:
import pandas as pd
ruta= '/content/supply_chain_logistics_delay_dataset.csv'
df= pd.read_csv(ruta)


In [2]:
df['delivery_delayed'] = df['delivery_delayed'].map({'Yes': 1, 'No': 0})
df['delivery_delayed'].value_counts()

,count
delivery_delayed,
0,74702
1,25298


In [3]:
nulos = df.isnull().sum()
nulos[nulos > 0].sort_values(ascending=False)

,0


In [4]:
cols_fecha = ['shipment_date', 'expected_delivery_date', 'actual_delivery_date']
df[cols_fecha] = df[cols_fecha].apply(pd.to_datetime)

# Verificar que se convirtieron bien
df[cols_fecha].dtypes

,0
shipment_date,datetime64[ns]
expected_delivery_date,datetime64[ns]
actual_delivery_date,datetime64[ns]


In [5]:
cols_booleanas = ['customs_required', 'customs_delay', 'documentation_complete']

df[cols_booleanas] = df[cols_booleanas].apply(lambda x: x.map({'Yes': 1, 'No': 0}))

# Verificar
df[cols_booleanas].dtypes

,0
customs_required,int64
customs_delay,int64
documentation_complete,int64


In [6]:
cols_categoricas = [
    'traffic_congestion_level', 'weather_condition', 'delay_reason',
    'supplier_country', 'warehouse_type', 'documentation_error',
    'transport_mode', 'vehicle_type', 'fuel_type',
    'route_type', 'road_condition', 'order_priority',
    'product_category', 'product_subcategory',
    'origin_country', 'origin_state', 'origin_city',
    'destination_country', 'destination_state', 'destination_city',
    'destination_region', 'origin_warehouse'
]

df[cols_categoricas] = df[cols_categoricas].astype('category')

# Verificar
df[cols_categoricas].dtypes

,0
traffic_congestion_level,category
weather_condition,category
delay_reason,category
supplier_country,category
warehouse_type,category
documentation_error,category
transport_mode,category
vehicle_type,category
fuel_type,category
route_type,category


In [7]:


# Tabla resumen: tipo, nulos y valores únicos por columna
resumen = pd.DataFrame({
    'tipo': df.dtypes.astype(str),
    'nulos': df.isnull().sum(),
    'unicos': df.nunique(),
    '%unicos': (df.nunique() / len(df) * 100).round(4)
}).sort_values('unicos')

pd.set_option('display.max_rows', None)
resumen

,tipo,nulos,unicos,%unicos
delivery_successful,bool,0,2,0.002
documentation_complete,int64,0,2,0.002
customs_delay,int64,0,2,0.002
delivery_delayed,int64,0,2,0.002
customs_required,int64,0,2,0.002
border_crossings,int64,0,3,0.003
risk_coverage_level,object,0,3,0.003
insurance_coverage,object,0,3,0.003
delivery_attempts,int64,0,3,0.003
traffic_congestion_level,category,0,3,0.003


In [8]:
df['insurance_coverage'] = df['insurance_coverage'].astype('category')
df['risk_coverage_level'] = df['risk_coverage_level'].astype('category')

In [9]:
# Verificar si destination_region y destination_state son idénticas
print("region vs state iguales:", (df['destination_region'] == df['destination_state']).all())

# Verificar si warehouse_id y origin_warehouse son idénticas
print("warehouse_id vs origin_warehouse iguales:", (df['warehouse_id'] == df['origin_warehouse']).all())

region vs state iguales: True
warehouse_id vs origin_warehouse iguales: True


In [10]:
# Eliminar columnas duplicadas (nos quedamos con destination_state y origin_warehouse)
df.drop(columns=['destination_region', 'warehouse_id'], inplace=True)

# Verificar
print(f"Columnas restantes: {df.shape[1]}")

Columnas restantes: 104


In [11]:
# Verificar duplicados (excluyendo los IDs, porque dos envíos distintos podrían tener datos iguales pero IDs diferentes)
cols_sin_ids = df.columns.difference(['shipment_id', 'order_id', 'tracking_number', 'product_id',
                                       'supplier_id', 'carrier_id', 'route_id'])

print(f"Duplicados considerando TODAS las columnas: {df.duplicated().sum()}")
print(f"Duplicados excluyendo IDs: {df.duplicated(subset=cols_sin_ids).sum()}")

Duplicados considerando TODAS las columnas: 0
Duplicados excluyendo IDs: 0


# === ELIMINACIÓN DE DUPLICADOS ===
 Se verificaron duplicados de dos formas:
1. Considerando todas las columnas → 0 duplicados
2. Excluyendo columnas ID (para detectar registros con contenido idéntico pero IDs distintos) → 0 duplicados
# Resultado: No se requiere eliminación de filas.

In [12]:
# Identificar las columnas numéricas (excluyendo las booleanas que ya convertimos)
cols_booleanas = ['delivery_delayed', 'customs_required', 'customs_delay', 'documentation_complete']
cols_numericas = df.select_dtypes(include=['int64', 'float64']).columns.difference(cols_booleanas).tolist()

print(f"Total variables numéricas: {len(cols_numericas)}")
print(cols_numericas)

Total variables numéricas: 66
['actual_delivery_days', 'average_speed_kmh', 'backorder_quantity', 'border_crossings', 'carrier_experience_years', 'carrier_rating', 'carrier_reliability_score', 'customs_cost', 'customs_processing_hours', 'customs_risk_score', 'delay_days', 'delay_hours', 'delivery_attempts', 'delivery_performance_score', 'destination_latitude', 'destination_longitude', 'distance_km', 'driver_experience_years', 'estimated_delivery_days', 'fuel_consumption_liters', 'fuel_cost', 'fuel_efficiency', 'fuel_price_per_liter', 'handling_cost', 'handling_time_hours', 'historical_delay_rate', 'humidity_percent', 'insurance_cost', 'inventory_level', 'inventory_turnover', 'loading_time_hours', 'number_of_stops', 'on_time_delivery_rate', 'order_backlog', 'order_quantity', 'order_value', 'origin_latitude', 'origin_longitude', 'package_volume_m3', 'package_weight_kg', 'packing_time_hours', 'picking_time_hours', 'processing_time_hours', 'rainfall_mm', 'route_efficiency', 'route_quality'

In [13]:
# Detección de outliers con método IQR para todas las numéricas
import numpy as np

resultados_outliers = []

for col in cols_numericas:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    limite_inf = Q1 - 1.5 * IQR
    limite_sup = Q3 + 1.5 * IQR

    outliers = df[(df[col] < limite_inf) | (df[col] > limite_sup)]
    n_outliers = len(outliers)
    pct = round(n_outliers / len(df) * 100, 2)

    resultados_outliers.append({
        'variable': col,
        'Q1': round(Q1, 2),
        'Q3': round(Q3, 2),
        'IQR': round(IQR, 2),
        'limite_inf': round(limite_inf, 2),
        'limite_sup': round(limite_sup, 2),
        'n_outliers': n_outliers,
        '%_outliers': pct
    })

df_outliers = pd.DataFrame(resultados_outliers).sort_values('%_outliers', ascending=False)

# Mostrar solo las que tienen outliers
pd.set_option('display.max_rows', None)
print(df_outliers[df_outliers['n_outliers'] > 0].to_string(index=False))
print(f"\nVariables SIN outliers: {len(df_outliers[df_outliers['n_outliers'] == 0])}")

                      variable       Q1        Q3       IQR  limite_inf  limite_sup  n_outliers  %_outliers
            backorder_quantity     0.00      0.00      0.00        0.00        0.00       23870       23.87
                    delay_days     0.00      1.00      1.00       -1.50        2.50       17998       18.00
                   delay_hours     0.00     26.00     26.00      -39.00       65.00       17998       18.00
                   rainfall_mm     0.00      5.20      5.20       -7.80       13.00       15087       15.09
                  customs_cost     0.00   3754.04   3754.04    -5631.07     9385.11       13076       13.08
              origin_longitude    11.58     73.14     61.55      -80.75      165.46       10062       10.06
         destination_longitude    11.58     73.14     61.55      -80.75      165.46        9995        9.99
                 stockout_risk     0.00      0.21      0.21       -0.32        0.52        5004        5.00
          total_logistics_co

In [14]:
cols_coordenadas = ['origin_latitude', 'origin_longitude',
                    'destination_latitude', 'destination_longitude']

df.drop(columns=cols_coordenadas, inplace=True)

print(f"Columnas restantes: {df.shape[1]}")

Columnas restantes: 100


In [15]:
# Recalcular outliers sin las coordenadas (ya eliminadas)

import plotly.express as px
import plotly.subplots as make_subplots
import plotly.graph_objects as go

cols_booleanas = ['delivery_delayed', 'customs_required', 'customs_delay', 'documentation_complete']
cols_numericas = df.select_dtypes(include=['int64', 'float64']).columns.difference(cols_booleanas).tolist()

resultados_outliers = []

for col in cols_numericas:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    limite_inf = Q1 - 1.5 * IQR
    limite_sup = Q3 + 1.5 * IQR

    outliers = df[(df[col] < limite_inf) | (df[col] > limite_sup)]
    n_outliers = len(outliers)
    pct = round(n_outliers / len(df) * 100, 2)

    resultados_outliers.append({
        'variable': col,
        'Q1': round(Q1, 2),
        'Q3': round(Q3, 2),
        'IQR': round(IQR, 2),
        'limite_inf': round(limite_inf, 2),
        'limite_sup': round(limite_sup, 2),
        'n_outliers': n_outliers,
        '%_outliers': pct
    })

df_outliers = pd.DataFrame(resultados_outliers).sort_values('%_outliers', ascending=False)

# Graficar top 12
top_outliers = df_outliers[df_outliers['n_outliers'] > 0].head(12)['variable'].tolist()

fig = make_subplots.make_subplots(rows=3, cols=4, subplot_titles=top_outliers)

for i, col in enumerate(top_outliers):
    row = i // 4 + 1
    col_pos = i % 4 + 1
    fig.add_trace(
        go.Box(y=df[col], name=col, marker_color='#2E75B6', showlegend=False),
        row=row, col=col_pos
    )

fig.update_layout(
    title_text='Boxplots — Top 12 Variables con Outliers (método IQR)',
    height=800, width=1100,
    template='plotly_white'
)
fig.show()

# === CONCLUSIÓN DE OUTLIERS ===
Se detectaron outliers en 24 de 62 variables numéricas mediante el método IQR.
 Decisión: NO se eliminan. Los outliers identificados corresponden a:
  - Variables target (delay_days, delay_hours): retrasos grandes = casos de mayor interés analítico
  - Variables de costo: distribución right-skewed natural en logística
  - backorder_quantity: 23.87% "outliers" indica que el IQR no es adecuado para esta distribución
  - Variables climáticas/operacionales: variación natural del negocio
Al ser un dataset sintético, no se esperan errores de captura que justifiquen eliminación.
Se conservan todos los registros para no sesgar el análisis ni perder información valiosa.

In [16]:
# === ELIMINACIÓN DE COLUMNAS IDENTIFICADORAS ===
# Estas columnas son IDs sin significado analítico propio.
# Ya tenemos variables que capturan la información útil de cada entidad:
#   - supplier_id → supplier_rating, supplier_reliability_score, supplier_country, etc.
#   - carrier_id → carrier_rating, carrier_reliability_score, carrier_experience_years, etc.
#   - route_id → route_type, route_quality, route_efficiency, distance_km, etc.
#   - product_id → product_category, product_subcategory, unit_price, etc.
#   - shipment_id, order_id, tracking_number → identificadores únicos por registro

cols_ids = ['shipment_id', 'order_id', 'tracking_number',
            'product_id', 'supplier_id', 'carrier_id', 'route_id']

df.drop(columns=cols_ids, inplace=True)

print(f"Columnas eliminadas: {len(cols_ids)}")
print(f"Columnas restantes: {df.shape[1]}")
print(f"Forma del dataset: {df.shape}")

Columnas eliminadas: 7
Columnas restantes: 93
Forma del dataset: (100000, 93)


# === HIPÓTESIS DE NEGOCIO ===
Planteadas ANTES del EDA, basadas en conocimiento del dominio de logística.

H1: "Las condiciones climáticas adversas (lluvia fuerte, nieve, tormenta) incrementan significativamente la probabilidad de retraso en la entrega."
Razón: El clima afecta la velocidad de transporte y genera desvíos de ruta.

H3: "Los proveedores con menor rating de confiabilidad (supplier_reliability_score) generan más retrasos en la cadena, independientemente del modo de transporte."
Razón: Un proveedor poco confiable entrega tarde al almacén, lo que retrasa
toda la cadena hacia el cliente final.

H6: "Los envíos con alta congestión de tráfico (traffic_congestion_level = Severe) combinados con condiciones de carretera pobres (road_condition = Poor/Severe) generan los retrasos más largos en horas."
Razón: Son dos factores externos que se potencian mutuamente — no es lo mismo
un tráfico severo en una autopista en buen estado que en una vía deteriorada.
Estas hipótesis se validarán durante el EDA con evidencia estadística y visual.

In [17]:
import pandas as pd

# === Variables seleccionadas por relevancia al problema de negocio ===
target_y_apoyo = ['delivery_delayed', 'delay_days', 'delay_hours', 'delivery_performance_score']

vars_clima = ['weather_condition', 'rainfall_mm', 'wind_speed_kmh', 'visibility_km']
vars_proveedor = ['supplier_reliability_score', 'supplier_rating', 'supplier_historical_delay_rate', 'supplier_lead_time_days']
vars_trafico_via = ['traffic_congestion_level', 'road_condition', 'average_speed_kmh']
vars_otras = ['carrier_rating', 'historical_delay_rate', 'on_time_delivery_rate',
              'warehouse_utilization', 'stockout_risk', 'customs_delay',
              'transport_mode', 'order_priority',
              'distance_km', 'number_of_stops', 'border_crossings']
cols_relevantes = target_y_apoyo + vars_clima + vars_proveedor + vars_trafico_via + vars_otras

# === 1. Distribución del target ===
print("=== DISTRIBUCIÓN delivery_delayed ===")
print(df['delivery_delayed'].value_counts())
print(df['delivery_delayed'].value_counts(normalize=True) * 100)

# === 2. Descriptivas numéricas (solo las relevantes) ===
numericas = df[cols_relevantes].select_dtypes(include=['int64', 'float64']).columns.tolist()
print("\n=== DESCRIPTIVAS NUMÉRICAS ===")
print(df[numericas].describe().T)

# === 3. Descriptivas categóricas (solo las relevantes) ===
categoricas = df[cols_relevantes].select_dtypes(include=['category', 'object']).columns.tolist()
print("\n=== DESCRIPTIVAS CATEGÓRICAS ===")
for col in categoricas:
    print(f"\n{col}:")
    print(df[col].value_counts())
    print(f"% distribución:\n{df[col].value_counts(normalize=True)*100}")

=== DISTRIBUCIÓN delivery_delayed ===
delivery_delayed
0    74702
1    25298
Name: count, dtype: int64
delivery_delayed
0    74.702
1    25.298
Name: proportion, dtype: float64

=== DESCRIPTIVAS NUMÉRICAS ===
                                   count         mean          std    min  \
delivery_delayed                100000.0     0.252980     0.434722   0.00   
delay_days                      100000.0     1.009420     2.006298   0.00   
delay_hours                     100000.0    26.494810    51.607328   0.00   
delivery_performance_score      100000.0    72.086804    13.646024  34.10   
rainfall_mm                     100000.0     6.115022    15.811097   0.00   
wind_speed_kmh                  100000.0    36.964300    21.603866   0.00   
visibility_km                   100000.0    10.511692     5.472654   1.00   
supplier_reliability_score      100000.0    77.079444    11.612446  43.50   
supplier_rating                 100000.0     3.859832     0.598741   2.15   
supplier_historical_d

In [18]:
import pandas as pd

# === H1: Clima vs retraso ===
print("=== H1: % de retraso por condición climática ===")
print(df.groupby('weather_condition')['delivery_delayed'].mean().sort_values(ascending=False) * 100)

print("\n=== H1 (apoyo): promedio de rainfall_mm — retrasados vs a tiempo ===")
print(df.groupby('delivery_delayed')['rainfall_mm'].mean())

# === H3: Confiabilidad del proveedor vs retraso ===
print("\n=== H3: correlación supplier_reliability_score vs delivery_delayed ===")
print(df[['supplier_reliability_score', 'delivery_delayed']].corr())

print("\n=== H3: % de retraso por cuartil de confiabilidad del proveedor ===")
df['supplier_reliability_cuartil'] = pd.qcut(df['supplier_reliability_score'], 4,
                                               labels=['Q1 (menor confiab.)', 'Q2', 'Q3', 'Q4 (mayor confiab.)'])
print(df.groupby('supplier_reliability_cuartil')['delivery_delayed'].mean() * 100)

# === H6: Tráfico severo + vía en mal estado vs horas de retraso ===
print("\n=== H6: promedio de delay_hours por combinación tráfico x vía ===")
tabla_h6 = df.groupby(['traffic_congestion_level', 'road_condition'])['delay_hours'].mean().unstack()
print(tabla_h6)

# === Resto de predictoras vs target (numéricas): correlación simple ===
print("\n=== Correlación de otras numéricas vs delivery_delayed ===")
otras_numericas = ['carrier_rating', 'historical_delay_rate', 'on_time_delivery_rate',
                    'warehouse_utilization', 'stockout_risk', 'distance_km',
                    'number_of_stops', 'border_crossings', 'average_speed_kmh',
                    'supplier_historical_delay_rate', 'supplier_lead_time_days']
print(df[otras_numericas + ['delivery_delayed']].corr()['delivery_delayed'].sort_values(ascending=False))

# === Resto de predictoras vs target (categóricas): % retraso por categoría ===
print("\n=== % retraso por transport_mode ===")
print(df.groupby('transport_mode')['delivery_delayed'].mean().sort_values(ascending=False) * 100)

print("\n=== % retraso por order_priority ===")
print(df.groupby('order_priority')['delivery_delayed'].mean().sort_values(ascending=False) * 100)

print("\n=== % retraso por customs_delay ===")
print(df.groupby('customs_delay')['delivery_delayed'].mean() * 100)

=== H1: % de retraso por condición climática ===
weather_condition
Storm           37.252964
Snow            30.771835
Heavy Rain      30.091928
Fog             27.474445
Extreme Heat    25.922953
Rain            25.826446
Cloudy          24.510938
Clear           23.066850
Name: delivery_delayed, dtype: float64

=== H1 (apoyo): promedio de rainfall_mm — retrasados vs a tiempo ===
delivery_delayed
0    6.095237
1    6.173445
Name: rainfall_mm, dtype: float64

=== H3: correlación supplier_reliability_score vs delivery_delayed ===
                            supplier_reliability_score  delivery_delayed
supplier_reliability_score                    1.000000         -0.254529
delivery_delayed                             -0.254529          1.000000

=== H3: % de retraso por cuartil de confiabilidad del proveedor ===
supplier_reliability_cuartil
Q1 (menor confiab.)    40.500136
Q2                     26.884698
Q3                     20.926285
Q4 (mayor confiab.)    12.380762
Name: delivery_d

/tmp/ipykernel_747/3507349101.py:5: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.

/tmp/ipykernel_747/3507349101.py:17: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.

/tmp/ipykernel_747/3507349101.py:21: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.

/tmp/ipykernel_747/3507349101.py:34: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False 

transport_mode
Courier       25.859072
Air           25.731695
Ship          25.483164
Multimodal    25.220459
Truck         25.160976
Van           25.142429
Rail          24.758939
Name: delivery_delayed, dtype: float64

=== % retraso por order_priority ===
order_priority
Low         25.785732
High        25.343365
Normal      25.181330
Critical    24.795613
Name: delivery_delayed, dtype: float64

=== % retraso por customs_delay ===
customs_delay
0    25.310587
1    25.190621
Name: delivery_delayed, dtype: float64


/tmp/ipykernel_747/3507349101.py:37: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



In [19]:
import pandas as pd

# === Confirmar efecto de tráfico solo sobre el target binario ===
print("=== % de retraso por nivel de congestión de tráfico ===")
print(df.groupby('traffic_congestion_level', observed=True)['delivery_delayed'].mean().sort_values(ascending=False) * 100)

print("\n=== % de retraso por condición de vía (sola) ===")
print(df.groupby('road_condition', observed=True)['delivery_delayed'].mean().sort_values(ascending=False) * 100)

# === Matriz de correlación ENTRE predictoras fuertes (multicolinealidad) ===
predictoras_fuertes = ['supplier_reliability_score', 'on_time_delivery_rate',
                        'supplier_historical_delay_rate', 'historical_delay_rate',
                        'carrier_rating', 'warehouse_utilization', 'stockout_risk']

print("\n=== Matriz de correlación entre predictoras fuertes ===")
print(df[predictoras_fuertes].corr().round(2))

=== % de retraso por nivel de congestión de tráfico ===
traffic_congestion_level
Severe      51.107570
Moderate    18.025058
Low          3.493010
Name: delivery_delayed, dtype: float64

=== % de retraso por condición de vía (sola) ===
road_condition
Poor         26.079473
Severe       26.025791
Fair         25.784541
Excellent    25.264321
Good         24.906885
Name: delivery_delayed, dtype: float64

=== Matriz de correlación entre predictoras fuertes ===
                                supplier_reliability_score  \
supplier_reliability_score                            1.00   
on_time_delivery_rate                                 0.00   
supplier_historical_delay_rate                       -0.96   
historical_delay_rate                                -0.00   
carrier_rating                                        0.00   
warehouse_utilization                                 0.00   
stockout_risk                                         0.00   

                                on_time_d

In [20]:
import pandas as pd

# === Segmentación: % de retraso por país de proveedor ===
print("=== % retraso por supplier_country ===")
print(df.groupby('supplier_country', observed=True)['delivery_delayed'].mean().sort_values(ascending=False) * 100)

# === Segmentación: % de retraso por país de destino ===
print("\n=== % retraso por destination_country ===")
print(df.groupby('destination_country', observed=True)['delivery_delayed'].mean().sort_values(ascending=False) * 100)

# === Segmentación cruzada: tráfico severo x país de destino (¿dónde se concentra el peor riesgo?) ===
print("\n=== % retraso en tráfico Severe, por país de destino ===")
df_severo = df[df['traffic_congestion_level'] == 'Severe']
print(df_severo.groupby('destination_country', observed=True)['delivery_delayed'].mean().sort_values(ascending=False) * 100)

# === Segmentación: cuartil de confiabilidad del proveedor x país del proveedor ===
print("\n=== % retraso por país de proveedor, solo en cuartil de menor confiabilidad ===")
df_q1 = df[df['supplier_reliability_cuartil'] == 'Q1 (menor confiab.)']
print(df_q1.groupby('supplier_country', observed=True)['delivery_delayed'].mean().sort_values(ascending=False) * 100)

=== % retraso por supplier_country ===
supplier_country
Malaysia      28.614120
India         26.348076
Bangladesh    26.060016
UAE           25.652410
Turkey        25.467943
USA           25.343954
Vietnam       25.093156
China         24.769295
Pakistan      23.730815
Germany       22.444137
Name: delivery_delayed, dtype: float64

=== % retraso por destination_country ===
destination_country
UAE         25.755913
UK          25.511022
USA         25.462731
China       25.321119
Germany     25.305061
Pakistan    25.174212
Turkey      24.927997
Name: delivery_delayed, dtype: float64

=== % retraso en tráfico Severe, por país de destino ===
destination_country
UK          51.978643
Germany     51.648000
China       51.634793
Turkey      51.115242
UAE         51.059257
USA         50.870761
Pakistan    50.693023
Name: delivery_delayed, dtype: float64

=== % retraso por país de proveedor, solo en cuartil de menor confiabilidad ===
supplier_country
UAE           41.655076
Malaysia      41

In [21]:
import sqlite3
import pandas as pd

# Crear conexión en memoria y cargar el DataFrame limpio como tabla SQL
conn = sqlite3.connect(":memory:")
df.to_sql("envios", conn, index=False, if_exists="replace")

# === Consulta 1: % de retraso por condición climática (valida H1) ===
q1 = """
SELECT weather_condition,
       COUNT(*) AS total_envios,
       SUM(delivery_delayed) AS retrasados,
       ROUND(100.0 * SUM(delivery_delayed) / COUNT(*), 2) AS pct_retraso
FROM envios
GROUP BY weather_condition
ORDER BY pct_retraso DESC;
"""
print("=== Q1: % retraso por condición climática ===")
print(pd.read_sql(q1, conn))

# === Consulta 2: Segmentos de confiabilidad del proveedor con CASE WHEN (valida H3) ===
q2 = """
SELECT
    CASE
        WHEN supplier_reliability_score < 69.4 THEN 'Q1 - Baja confiabilidad'
        WHEN supplier_reliability_score < 77.8 THEN 'Q2'
        WHEN supplier_reliability_score < 84.7 THEN 'Q3'
        ELSE 'Q4 - Alta confiabilidad'
    END AS segmento_confiabilidad,
    COUNT(*) AS total_envios,
    ROUND(100.0 * SUM(delivery_delayed) / COUNT(*), 2) AS pct_retraso
FROM envios
GROUP BY segmento_confiabilidad
ORDER BY pct_retraso DESC;
"""
print("\n=== Q2: % retraso por segmento de confiabilidad del proveedor ===")
print(pd.read_sql(q2, conn))

# === Consulta 3: Horas de retraso promedio por tráfico x condición de vía ===
q3 = """
SELECT traffic_congestion_level,
       road_condition,
       ROUND(AVG(delay_hours), 2) AS promedio_horas_retraso,
       COUNT(*) AS total_envios
FROM envios
GROUP BY traffic_congestion_level, road_condition
ORDER BY promedio_horas_retraso DESC;
"""
print("\n=== Q3: Horas de retraso promedio por tráfico x condición de vía ===")
print(pd.read_sql(q3, conn))

# === Consulta 4: Países con mayor % de retraso, con volumen significativo (HAVING) ===
q4 = """
SELECT supplier_country,
       COUNT(*) AS total_envios,
       ROUND(100.0 * SUM(delivery_delayed) / COUNT(*), 2) AS pct_retraso
FROM envios
GROUP BY supplier_country
HAVING COUNT(*) >= 500
ORDER BY pct_retraso DESC;
"""
print("\n=== Q4: % retraso por país del proveedor (mín. 500 envíos) ===")
print(pd.read_sql(q4, conn))

# === Consulta 5: Proveedor Y transportista ambos por debajo del promedio (subconsulta) ===
q5 = """
SELECT COUNT(*) AS total_envios,
       ROUND(100.0 * SUM(delivery_delayed) / COUNT(*), 2) AS pct_retraso
FROM envios
WHERE supplier_reliability_score < (SELECT AVG(supplier_reliability_score) FROM envios)
  AND carrier_rating < (SELECT AVG(carrier_rating) FROM envios);
"""
print("\n=== Q5: % retraso cuando proveedor Y transportista están ambos por debajo del promedio ===")
print(pd.read_sql(q5, conn))

=== Q1: % retraso por condición climática ===
  weather_condition  total_envios  retrasados  pct_retraso
0             Storm          4048        1508        37.25
1              Snow          2954         909        30.77
2        Heavy Rain          6962        2095        30.09
3               Fog          4011        1102        27.47
4      Extreme Heat          4984        1292        25.92
5              Rain         12100        3125        25.83
6            Cloudy         19885        4874        24.51
7             Clear         45056       10393        23.07

=== Q2: % retraso por segmento de confiabilidad del proveedor ===
    segmento_confiabilidad  total_envios  pct_retraso
0  Q1 - Baja confiabilidad         24831        40.85
1                       Q2         24890        27.12
2                       Q3         25154        21.02
3  Q4 - Alta confiabilidad         25125        12.41

=== Q3: Horas de retraso promedio por tráfico x condición de vía ===
   traffic_conge

### Conclusiones de las Consultas SQL

**Q1 — % de retraso por condición climática:**
Las condiciones climáticas severas (Storm, Snow, Heavy Rain) presentan tasas de retraso entre 30% y 37%,
superando significativamente el promedio general de 25.30%. En contraste, el clima despejado (Clear)
tiene la menor tasa con 23.07%. Esto confirma que el clima adverso es un factor de riesgo real,
aunque no el más determinante de todos los analizados.

**Q2 — % de retraso por segmento de confiabilidad del proveedor:**
Los proveedores de baja confiabilidad (Q1) acumulan un 40.85% de retraso, más de 3 veces
lo que registran los de alta confiabilidad (Q4: 12.41%).
**Q3 — Horas de retraso promedio por tráfico × condición de vía:**
El nivel de congestión de tráfico domina completamente el efecto sobre las horas de retraso:
Low (~3-4h), Moderate (~18-21h), Severe (~52-54h). Dentro de cada nivel de tráfico,
la condición de la vía apenas varía 1-2 horas, confirmando que la vía no potencia
el efecto del tráfico como se planteó en H6.

**Q4 — % de retraso por país del proveedor:**
Todos los países de origen cuentan con más de 500 envíos (muestra suficiente gracias al filtro HAVING),
y el rango de retraso va de 22.44% (Alemania) a 28.61% (Malasia). Esta diferencia de ~6 puntos
porcentuales es modesta comparada con los factores operativos (tráfico: 48 pp, proveedor: 28 pp),
confirmando que la geografía no es una palanca útil de segmentación.

**Q5 — Riesgo combinado proveedor + transportista por debajo del promedio:**
El hallazgo más accionable de todo el análisis: cuando tanto el proveedor como el transportista
están por debajo del promedio, la tasa de retraso sube a 43.15% (vs 25.30% general, un +70%).
Este segmento agrupa 23,384 envíos (23.4% del total) y representa el foco prioritario
de intervención operativa para la empresa.

In [37]:
import plotly.graph_objects as go

fig = go.Figure()

# A tiempo
fig.add_trace(go.Bar(
    y=['Envios'],
    x=[74.70],
    orientation='h',
    name='A tiempo',
    marker=dict(color='#2E86AB'),
    text=['74.70%  A tiempo'],
    textposition='inside',
    textfont=dict(size=16, family='Calibri', color='white'),
    hovertemplate='A tiempo: 74,702 envios (74.70%)<extra></extra>'
))

# Retrasados
fig.add_trace(go.Bar(
    y=['Envios'],
    x=[25.30],
    orientation='h',
    name='Retrasados',
    marker=dict(color='#E84855'),
    text=['25.30%  Retrasados'],
    textposition='inside',
    textfont=dict(size=16, family='Calibri', color='white'),
    hovertemplate='Retrasados: 25,298 envios (25.30%)<extra></extra>'
))

fig.update_layout(
    barmode='stack',
    title=dict(
        text='1 de cada 4 envios llega tarde<br><sup>Distribucion de delivery_delayed (n=100,000 envios)</sup>',
        font=dict(size=20, family='Calibri', color='#1B3A5C'),
        x=0.5
    ),
    xaxis=dict(
        showticklabels=False,
        showgrid=False,
        zeroline=False
    ),
    yaxis=dict(
        showticklabels=False
    ),
    annotations=[
        dict(
            text='74,702 envios a tiempo  |  25,298 envios retrasados',
            x=0.5, y=-0.15,
            xref='paper', yref='paper',
            font=dict(size=12, family='Calibri', color='#555555'),
            showarrow=False
        )
    ],
    width=750, height=250,
    paper_bgcolor='white',
    plot_bgcolor='white',
    margin=dict(t=90, b=60, l=40, r=40),
    showlegend=False
)

fig.show()

In [33]:
import pandas as pd

# === PERFIL DE LOS ENVIOS RETRASADOS ===
retrasados = df[df['delivery_delayed'] == 1]
print(f"Total envios retrasados: {len(retrasados):,} de {len(df):,} ({len(retrasados)/len(df)*100:.2f}%)")

print("\n" + "=" * 60)
print("CUANTOS DIAS SE RETRASAN?")
print("=" * 60)
print(retrasados['delay_days'].value_counts().sort_index())
print(f"\nModa: {retrasados['delay_days'].mode()[0]} dias (el retraso mas frecuente)")
print(f"Media: {retrasados['delay_days'].mean():.2f} dias")
print(f"Mediana: {retrasados['delay_days'].median():.0f} dias")

print("\n" + "=" * 60)
print("CUANTAS HORAS SE RETRASAN?")
print("=" * 60)
print(retrasados['delay_hours'].describe().round(1))
print(f"\nRango: de {retrasados['delay_hours'].min()} a {retrasados['delay_hours'].max()} horas")

print("\n" + "=" * 60)
print("POR QUE SE RETRASAN? (delay_reason)")
print("=" * 60)
print(retrasados['delay_reason'].value_counts())
print(f"\n% por razon:")
print((retrasados['delay_reason'].value_counts(normalize=True)*100).round(2))

print("\n" + "=" * 60)
print("RETRASOS EN EL TIEMPO")
print("=" * 60)
df['shipment_date'] = pd.to_datetime(df['shipment_date'])
df['anio'] = df['shipment_date'].dt.year
df['mes'] = df['shipment_date'].dt.month
df['anio_mes'] = df['shipment_date'].dt.to_period('M')

print("\n--- % de retraso por anio ---")
print(df.groupby('anio', observed=True)['delivery_delayed'].agg(['sum','count','mean']).rename(
    columns={'sum':'retrasados','count':'total','mean':'pct_retraso'}).assign(
    pct_retraso=lambda x: (x['pct_retraso']*100).round(2)))

print("\n--- % de retraso por mes (todos los anios) ---")
print(df.groupby('mes', observed=True)['delivery_delayed'].agg(['sum','count','mean']).rename(
    columns={'sum':'retrasados','count':'total','mean':'pct_retraso'}).assign(
    pct_retraso=lambda x: (x['pct_retraso']*100).round(2)))

Total envios retrasados: 25,298 de 100,000 (25.30%)

CUANTOS DIAS SE RETRASAN?
delay_days
1    3668
2    3632
3    3612
4    3589
5    3562
6    3637
7    3598
Name: count, dtype: int64

Moda: 1 dias (el retraso mas frecuente)
Media: 3.99 dias
Mediana: 4 dias

CUANTAS HORAS SE RETRASAN?
count    25298.0
mean       104.7
std         48.3
min         25.0
25%         61.0
50%        105.0
75%        149.0
max        185.0
Name: delay_hours, dtype: float64

Rango: de 25 a 185 horas

POR QUE SE RETRASAN? (delay_reason)
delay_reason
Traffic                 7067
Weather                 4128
Warehouse Congestion    3100
Supplier Delay          2994
Carrier Issue           2022
Customs                 2016
Documentation Error     1019
Vehicle Breakdown       1013
Inventory Shortage       988
Route Disruption         463
Labor Shortage           249
Port Congestion          239
No Delay                   0
Name: count, dtype: int64

% por razon:
delay_reason
Traffic                 27.94
Weathe

In [35]:
import plotly.graph_objects as go

razones = ['Customs', 'Carrier Issue', 'Supplier Delay',
           'Warehouse Congestion', 'Weather', 'Traffic']
pct = [7.97, 7.99, 11.83, 12.25, 16.32, 27.94]
colores = ['#BFBFBF', '#BFBFBF', '#2E86AB', '#2E86AB', '#F39C12', '#E84855']

fig = go.Figure()

fig.add_trace(go.Bar(
    y=razones,
    x=pct,
    orientation='h',
    marker=dict(color=colores, line=dict(color='white', width=1.5)),
    text=[f'{v}%' for v in pct],
    textposition='outside',
    textfont=dict(size=13, family='Calibri', color='#1B3A5C'),
    hovertemplate='%{y}<br>%{x}% de los retrasos<extra></extra>'
))

fig.update_layout(
    title=dict(
        text='Por que se retrasan los envios?<br><sup>Top 6 razones de retraso (solo envios retrasados, n=25,298)</sup>',
        font=dict(size=20, family='Calibri', color='#1B3A5C'),
        x=0.5
    ),
    xaxis=dict(
        title='% de los envios retrasados',
        titlefont=dict(size=12, family='Calibri', color='#555555'),
        tickfont=dict(size=11, family='Calibri'),
        range=[0, 33],
        showgrid=True, gridcolor='#F0F0F0'
    ),
    yaxis=dict(
        tickfont=dict(size=13, family='Calibri', color='#1B3A5C')
    ),
    annotations=[
        dict(
            text='Estas 6 causas concentran el 84% de todos los retrasos',
            x=0.5, y=-0.13,
            xref='paper', yref='paper',
            font=dict(size=11, family='Calibri', color='#555555'),
            showarrow=False
        )
    ],
    width=750, height=420,
    paper_bgcolor='white',
    plot_bgcolor='white',
    margin=dict(t=100, b=80, l=180, r=40),
    showlegend=False
)

fig.show()

#### Perfil de los envíos retrasados

De los 100,000 envíos analizados, 25,298 (25.30%) llegaron retrasados. Al analizar su perfil:

- **Días de retraso:** distribución uniforme entre 1 y 7 días (~3,600 envíos en cada nivel).
  No hay un pico dominante — cuando un envío se retrasa, puede tardar de 1 a 7 días
  con probabilidad similar. Media: 4 días. Rango en horas: 25 a 185.
- **Razones de retraso:** las 4 causas principales concentran el 68% de todos los retrasos:
  Tráfico (27.94%), Clima (16.32%), Congestión de almacén (12.25%) y Retraso del proveedor (11.83%).
  Esto confirma desde la variable `delay_reason` los mismos factores que identificamos
  de forma independiente en el EDA.
- **Comportamiento temporal:** la tasa de retraso es prácticamente constante en todos los años
  (24.95% a 25.59%) y todos los meses (24.70% a 25.67%). No hay estacionalidad.
  El problema es estructural y permanente, no se resuelve esperando un "mejor mes".

In [36]:
import plotly.graph_objects as go

dias = ['1 dia', '2 dias', '3 dias', '4 dias', '5 dias', '6 dias', '7 dias']
cantidades = [3668, 3632, 3612, 3589, 3562, 3637, 3598]

fig = go.Figure()

fig.add_trace(go.Bar(
    x=dias,
    y=cantidades,
    marker=dict(
        color=['#FDEDEC', '#F9C4C4', '#F1948A', '#E84855', '#E84855', '#C0392B', '#922B21'],
        line=dict(color='white', width=1.5)
    ),
    text=[f'{v:,}' for v in cantidades],
    textposition='outside',
    textfont=dict(size=13, family='Calibri', color='#1B3A5C'),
    hovertemplate='%{x}<br>Envios: %{y:,}<extra></extra>'
))

fig.update_layout(
    title=dict(
        text='Cuando se retrasan, cuantos dias tardan de mas?<br><sup>Distribucion de dias de retraso (solo envios retrasados, n=25,298)</sup>',
        font=dict(size=20, family='Calibri', color='#1B3A5C'),
        x=0.5
    ),
    xaxis=dict(
        tickfont=dict(size=13, family='Calibri', color='#1B3A5C')
    ),
    yaxis=dict(
        title='Cantidad de envios',
        titlefont=dict(size=12, family='Calibri', color='#555555'),
        tickfont=dict(size=11, family='Calibri'),
        showgrid=True, gridcolor='#F0F0F0'
    ),
    annotations=[
        dict(
            text='Distribucion uniforme: no hay un pico dominante. Media: 4 dias | Rango: 25 a 185 horas',
            x=0.5, y=-0.15,
            xref='paper', yref='paper',
            font=dict(size=11, family='Calibri', color='#555555'),
            showarrow=False
        )
    ],
    width=750, height=420,
    paper_bgcolor='white',
    plot_bgcolor='white',
    margin=dict(t=100, b=80, l=60, r=40),
    showlegend=False
)

fig.show()

#### Proporción retrasados vs a tiempo

De cada 4 envíos que salen del almacén, 1 llega tarde. La proporción 74.70% a tiempo
vs 25.30% retrasados representa un desbalance moderado que confirma que el problema
no es marginal: afecta a 1 de cada 4 operaciones de la cadena de suministro.

In [41]:
import plotly.graph_objects as go

categorias = ['Promedio general<br>(todos los envios)', 'Proveedor + Transportista<br>ambos por debajo del promedio']
valores = [25.30, 43.15]
colores = ['#2E86AB', '#E84855']

fig = go.Figure()

fig.add_trace(go.Bar(
    x=categorias,
    y=valores,
    marker=dict(color=colores, line=dict(color='white', width=1.5)),
    text=['25.30%', '43.15%'],
    textposition='outside',
    textfont=dict(size=18, family='Calibri', color='#1B3A5C'),
    width=0.5,
    hovertemplate='%{x}<br>Retraso: %{y}%<extra></extra>'
))

# Flecha con el +70%
fig.add_annotation(
    x=1, y=43.15,
    ax=0, ay=25.30,
    xref='x', yref='y',
    axref='x', ayref='y',
    showarrow=True,
    arrowhead=2,
    arrowsize=1.5,
    arrowwidth=2,
    arrowcolor='#E84855'
)

fig.add_annotation(
    x=0.5, y=35,
    text='<b>+17.85 pp</b><br>de diferencia',
    font=dict(size=16, family='Calibri', color='#E84855'),
    showarrow=False
)

fig.update_layout(
    title=dict(
        text='Cuando proveedor y transportista fallan juntos, el riesgo se dispara<br><sup>% de envios retrasados: promedio general vs segmento de riesgo combinado</sup>',
        font=dict(size=20, family='Calibri', color='#1B3A5C'),
        x=0.5
    ),
    yaxis=dict(
        title='% de envios retrasados',
        titlefont=dict(size=12, family='Calibri', color='#555555'),
        tickfont=dict(size=11, family='Calibri'),
        range=[0, 52],
        showgrid=True, gridcolor='#F0F0F0'
    ),
    xaxis=dict(
        tickfont=dict(size=12, family='Calibri', color='#1B3A5C')
    ),
    annotations=fig.layout.annotations + (
        dict(
            text='23,384 envios (23.4% del total) caen en este segmento de alto riesgo',
            x=0.5, y=-0.18,
            xref='paper', yref='paper',
            font=dict(size=12, family='Calibri', color='#555555'),
            showarrow=False
        ),
    ),
    width=700, height=480,
    paper_bgcolor='white',
    plot_bgcolor='white',
    margin=dict(t=100, b=80, l=60, r=40),
    showlegend=False
)

fig.show()

#### Riesgo combinado proveedor + transportista

Cuando tanto el proveedor como el transportista están por debajo del promedio,
la tasa de retraso sube de 25.30% a 43.15% — una diferencia de +17.85 puntos porcentuales.
Este segmento agrupa 23,384 envíos (23.4% del total) y representa el foco prioritario
de intervención operativa: es donde renegociar contratos o reasignar transportistas
tendría el mayor impacto.

In [23]:
import plotly.graph_objects as go

# Datos ordenados de menor a mayor riesgo
niveles = ['Low', 'Moderate', 'Severe']
pct_retraso = [3.49, 18.03, 51.11]
colores = ['#27AE60', '#F39C12', '#E84855']  # verde, amarillo, rojo

fig = go.Figure()

fig.add_trace(go.Bar(
    y=niveles,
    x=pct_retraso,
    orientation='h',
    marker=dict(
        color=colores,
        line=dict(color='white', width=1.5)
    ),
    text=[f'{v}%' for v in pct_retraso],
    textposition='outside',
    textfont=dict(size=15, family='Calibri', color='#1B3A5C'),
    hovertemplate='Congestión: %{y}<br>Retraso: %{x}%<extra></extra>'
))

# Línea de referencia: promedio general
fig.add_vline(
    x=25.30, line_dash='dash', line_color='#7F8C8D', line_width=1.5,
    annotation_text='Promedio general: 25.30%',
    annotation_position='top',
    annotation_font=dict(size=11, color='#7F8C8D', family='Calibri')
)

fig.update_layout(
    title=dict(
        text='El tráfico es el factor externo #1 de retraso<br><sup>% de envíos retrasados por nivel de congestión de tráfico</sup>',
        font=dict(size=20, family='Calibri', color='#1B3A5C'),
        x=0.5
    ),
    xaxis=dict(
        title='% de envíos retrasados',
        titlefont=dict(size=13, family='Calibri', color='#555555'),
        tickfont=dict(size=12, family='Calibri'),
        range=[0, 60],
        showgrid=True, gridcolor='#F0F0F0'
    ),
    yaxis=dict(
        tickfont=dict(size=14, family='Calibri', color='#1B3A5C'),
        categoryorder='array',
        categoryarray=niveles
    ),
    width=700, height=380,
    paper_bgcolor='white',
    plot_bgcolor='white',
    margin=dict(t=100, b=60, l=100, r=60),
    showlegend=False
)

fig.show()

In [38]:
import plotly.graph_objects as go

cuartiles = ['Q1<br>Baja confiabilidad', 'Q2', 'Q3', 'Q4<br>Alta confiabilidad']
pct_retraso = [40.50, 26.88, 20.93, 12.38]
colores = ['#E84855', '#F39C12', '#F9C846', '#27AE60']

fig = go.Figure()

fig.add_trace(go.Bar(
    x=cuartiles,
    y=pct_retraso,
    marker=dict(color=colores, line=dict(color='white', width=1.5)),
    text=[f'{v}%' for v in pct_retraso],
    textposition='outside',
    textfont=dict(size=14, family='Calibri', color='#1B3A5C'),
    hovertemplate='%{x}<br>Retraso: %{y}%<extra></extra>'
))

fig.add_hline(
    y=25.30, line_dash='dash', line_color='#7F8C8D', line_width=1.5,
    annotation_text='Promedio general: 25.30%',
    annotation_position='top right',
    annotation_font=dict(size=11, color='#7F8C8D', family='Calibri')
)

fig.update_layout(
    title=dict(
        text='A menor confiabilidad del proveedor, mas retrasos<br><sup>% de envios retrasados por cuartil de supplier_reliability_score</sup>',
        font=dict(size=20, family='Calibri', color='#1B3A5C'),
        x=0.5
    ),
    yaxis=dict(
        title='% de envios retrasados',
        titlefont=dict(size=12, family='Calibri', color='#555555'),
        tickfont=dict(size=11, family='Calibri'),
        range=[0, 48],
        showgrid=True, gridcolor='#F0F0F0'
    ),
    xaxis=dict(
        tickfont=dict(size=12, family='Calibri', color='#1B3A5C')
    ),
    width=700, height=450,
    paper_bgcolor='white',
    plot_bgcolor='white',
    margin=dict(t=100, b=60, l=60, r=40),
    showlegend=False
)

fig.show()

#### Confiabilidad del proveedor vs retraso

La confiabilidad del proveedor es el factor interno #1 de retraso. El análisis por cuartiles
muestra una escalera descendente clara:

- Q1 (baja confiabilidad): 40.50% de retraso — 4 de cada 10 envíos llegan tarde.
- Q4 (alta confiabilidad): 12.38% de retraso — apenas 1 de cada 10.
- La diferencia entre el peor y el mejor cuartil es de 28.12 puntos porcentuales.
  Este patrón es independiente del modo de transporte y del país de destino.

In [25]:
import plotly.graph_objects as go

# Datos ordenados de mayor a menor retraso
condiciones = ['Storm', 'Snow', 'Heavy Rain', 'Fog', 'Extreme Heat', 'Rain', 'Cloudy', 'Clear']
pct_retraso = [37.25, 30.77, 30.09, 27.47, 25.92, 25.83, 24.51, 23.07]
colores = ['#C0392B', '#D35400', '#E67E22', '#F39C12', '#F9C846', '#F9C846', '#7DCEA0', '#27AE60']

fig = go.Figure()

fig.add_trace(go.Bar(
    x=condiciones,
    y=pct_retraso,
    marker=dict(
        color=colores,
        line=dict(color='white', width=1.5)
    ),
    text=[f'{v}%' for v in pct_retraso],
    textposition='outside',
    textfont=dict(size=13, family='Calibri', color='#1B3A5C'),
    hovertemplate='%{x}<br>Retraso: %{y}%<extra></extra>'
))

# Línea de referencia: promedio general
fig.add_hline(
    y=25.30, line_dash='dash', line_color='#7F8C8D', line_width=1.5,
    annotation_text='Promedio general: 25.30%',
    annotation_position='top right',
    annotation_font=dict(size=11, color='#7F8C8D', family='Calibri')
)

fig.update_layout(
    title=dict(
        text='Las tormentas casi duplican el riesgo de retraso vs clima despejado<br><sup>% de envíos retrasados por condición climática</sup>',
        font=dict(size=20, family='Calibri', color='#1B3A5C'),
        x=0.5
    ),
    xaxis=dict(
        tickfont=dict(size=12, family='Calibri', color='#555555'),
        categoryorder='array',
        categoryarray=condiciones
    ),
    yaxis=dict(
        title='% de envíos retrasados',
        titlefont=dict(size=13, family='Calibri', color='#555555'),
        tickfont=dict(size=12, family='Calibri'),
        range=[0, 43],
        showgrid=True, gridcolor='#F0F0F0'
    ),
    width=750, height=450,
    paper_bgcolor='white',
    plot_bgcolor='white',
    margin=dict(t=100, b=60, l=60, r=40),
    showlegend=False
)

fig.show()

In [27]:
import plotly.figure_factory as ff
import pandas as pd
import numpy as np

predictoras = ['supplier_reliability_score', 'on_time_delivery_rate',
               'supplier_historical_delay_rate', 'historical_delay_rate',
               'carrier_rating', 'warehouse_utilization', 'stockout_risk']

etiquetas = ['Confiab. Proveedor', 'Puntualidad Transp.', 'Hist. Retraso Prov.',
             'Hist. Retraso Transp.', 'Rating Transp.', 'Utilizac. Almacen', 'Riesgo Stockout']

corr = df[predictoras].corr().round(2)
z = corr.values.tolist()

fig = ff.create_annotated_heatmap(
    z=z,
    x=etiquetas,
    y=etiquetas,
    colorscale=[
        [0.0, '#E84855'],
        [0.25, '#F9C846'],
        [0.5, '#FFFFFF'],
        [0.75, '#7DCEA0'],
        [1.0, '#2E86AB']
    ],
    zmin=-1, zmax=1,
    showscale=True,
    font_colors=['#1B3A5C', '#1B3A5C']
)

# Agregar subtitulo sin romper las anotaciones existentes
anotaciones = list(fig.layout.annotations)
anotaciones.append(dict(
    text='Los bloques de color intenso son variables redundantes entre si',
    x=0.5, y=1.15,
    xref='paper', yref='paper',
    font=dict(size=13, family='Calibri', color='#7F8C8D'),
    showarrow=False
))

fig.update_layout(
    title=dict(
        text='No hay 7 factores independientes, hay 5 palancas reales',
        font=dict(size=20, family='Calibri', color='#1B3A5C'),
        x=0.5,
        y=0.98
    ),
    annotations=anotaciones,
    xaxis=dict(tickfont=dict(size=11, family='Calibri'), tickangle=35, side='bottom'),
    yaxis=dict(tickfont=dict(size=11, family='Calibri'), autorange='reversed'),
    width=750, height=650,
    paper_bgcolor='white',
    margin=dict(t=140, b=130, l=160, r=40)
)

fig.update_traces(hovertemplate='%{x} vs %{y}<br>Correlacion: %{z}<extra></extra>')

fig.show()

In [43]:
import pandas as pd

# Dividir warehouse_utilization en rangos de 10%
df['rango_utilizacion'] = pd.cut(df['warehouse_utilization'],
    bins=[0, 40, 50, 60, 70, 80, 85, 90, 95, 100],
    labels=['35-40%', '40-50%', '50-60%', '60-70%', '70-80%', '80-85%', '85-90%', '90-95%', '95-100%'])

print("=== % de retraso por rango de utilizacion del almacen ===")
tabla = df.groupby('rango_utilizacion', observed=True)['delivery_delayed'].agg(['sum','count','mean'])
tabla.columns = ['retrasados', 'total', 'pct_retraso']
tabla['pct_retraso'] = (tabla['pct_retraso'] * 100).round(2)
print(tabla)

print("\n=== Estadisticas de warehouse_utilization ===")
print(f"Percentil 25: {df['warehouse_utilization'].quantile(0.25)}")
print(f"Percentil 50: {df['warehouse_utilization'].quantile(0.50)}")
print(f"Percentil 75: {df['warehouse_utilization'].quantile(0.75)}")
print(f"Percentil 90: {df['warehouse_utilization'].quantile(0.90)}")

=== % de retraso por rango de utilizacion del almacen ===
                   retrasados  total  pct_retraso
rango_utilizacion                                
35-40%                    194   2012         9.64
40-50%                    751   6094        12.32
50-60%                   1928  12064        15.98
60-70%                   2798  13910        20.12
70-80%                   8358  33071        25.27
80-85%                   2764   8979        30.78
85-90%                   4242  12874        32.95
90-95%                   1475   4022        36.67
95-100%                  2788   6974        39.98

=== Estadisticas de warehouse_utilization ===
Percentil 25: 65.1
Percentil 50: 74.2
Percentil 75: 84.7
Percentil 90: 91.3


#### Umbral de almacén saturado (>84.7%)

El umbral del percentil 75 (84.7%) no fue una elección arbitraria.
El análisis por rangos de utilización demuestra un punto de quiebre claro:

- **Por debajo del 80%:** la tasa de retraso crece gradualmente (9.64% → 25.27%).
- **A partir del 80-85%:** la tasa salta a 30.78% y sigue acelerándose hasta 39.98%
  en almacenes con 95-100% de utilización.
- El percentil 75 cae exactamente en 84.7%, coincidiendo con el punto donde
  la tasa de retraso se acelera.

Esto le da al negocio un umbral operativo concreto y sustentado:
cuando un almacén supera el 84.7% de utilización, el riesgo de retraso
pasa de niveles promedio (~25%) a niveles críticos (>30%).

In [44]:
import plotly.graph_objects as go

rangos = ['35-40%', '40-50%', '50-60%', '60-70%', '70-80%', '80-85%', '85-90%', '90-95%', '95-100%']
pct_retraso = [9.64, 12.32, 15.98, 20.12, 25.27, 30.78, 32.95, 36.67, 39.98]

# Colores: azul antes del umbral, rojo despues
colores = ['#2E86AB', '#2E86AB', '#2E86AB', '#2E86AB', '#2E86AB', '#E84855', '#E84855', '#E84855', '#E84855']

fig = go.Figure()

# Barras
fig.add_trace(go.Bar(
    x=rangos,
    y=pct_retraso,
    marker=dict(color=colores, line=dict(color='white', width=1.5)),
    text=[f'{v}%' for v in pct_retraso],
    textposition='outside',
    textfont=dict(size=12, family='Calibri', color='#1B3A5C'),
    hovertemplate='Utilizacion: %{x}<br>Retraso: %{y}%<extra></extra>'
))

# Linea del promedio general
fig.add_hline(
    y=25.30, line_dash='dash', line_color='#7F8C8D', line_width=1.5,
    annotation_text='Promedio general: 25.30%',
    annotation_position='top right',
    annotation_font=dict(size=10, color='#7F8C8D', family='Calibri')
)

# Linea vertical del umbral P75
fig.add_vline(
    x=4.5, line_dash='dot', line_color='#E84855', line_width=2,
    annotation_text='Umbral P75: 84.7%',
    annotation_position='top',
    annotation_font=dict(size=11, color='#E84855', family='Calibri')
)

fig.update_layout(
    title=dict(
        text='Por que definimos "almacen saturado" por encima del 84.7%?<br><sup>% de envios retrasados por rango de utilizacion del almacen</sup>',
        font=dict(size=20, family='Calibri', color='#1B3A5C'),
        x=0.5
    ),
    yaxis=dict(
        title='% de envios retrasados',
        titlefont=dict(size=12, family='Calibri', color='#555555'),
        tickfont=dict(size=11, family='Calibri'),
        range=[0, 47],
        showgrid=True, gridcolor='#F0F0F0'
    ),
    xaxis=dict(
        title='Rango de utilizacion del almacen',
        titlefont=dict(size=12, family='Calibri', color='#555555'),
        tickfont=dict(size=10, family='Calibri', color='#1B3A5C')
    ),
    annotations=list(fig.layout.annotations) + [
        dict(
            text='Por debajo del 80%: retraso crece gradualmente  |  Por encima del 84.7% (P75): retraso se acelera de 30% a 40%',
            x=0.5, y=-0.2,
            xref='paper', yref='paper',
            font=dict(size=11, family='Calibri', color='#555555'),
            showarrow=False
        )
    ],
    width=800, height=480,
    paper_bgcolor='white',
    plot_bgcolor='white',
    margin=dict(t=100, b=90, l=60, r=40),
    showlegend=False
)

fig.show()

## FEATURE ENGINEERING

In [28]:
import pandas as pd

# Crear la variable: 1 cuando AMBOS están por debajo del promedio
promedio_supplier = df['supplier_reliability_score'].mean()
promedio_carrier = df['carrier_rating'].mean()

print(f"Promedio supplier_reliability_score: {promedio_supplier:.2f}")
print(f"Promedio carrier_rating: {promedio_carrier:.2f}")

df['riesgo_proveedor_transportista'] = (
    (df['supplier_reliability_score'] < promedio_supplier) &
    (df['carrier_rating'] < promedio_carrier)
).astype(int)

# === Validación contra el target ===
print("\n=== Distribución de la nueva variable ===")
print(df['riesgo_proveedor_transportista'].value_counts())
print(f"\n% del total: {df['riesgo_proveedor_transportista'].mean()*100:.2f}% de los envios caen en riesgo combinado")

print("\n=== % de retraso por riesgo_proveedor_transportista ===")
print(df.groupby('riesgo_proveedor_transportista', observed=True)['delivery_delayed'].mean() * 100)

print("\n=== Promedio de delay_hours por riesgo_proveedor_transportista ===")
print(df.groupby('riesgo_proveedor_transportista', observed=True)['delay_hours'].mean().round(2))

Promedio supplier_reliability_score: 77.08
Promedio carrier_rating: 4.11

=== Distribución de la nueva variable ===
riesgo_proveedor_transportista
0    76616
1    23384
Name: count, dtype: int64

% del total: 23.38% de los envios caen en riesgo combinado

=== % de retraso por riesgo_proveedor_transportista ===
riesgo_proveedor_transportista
0    19.849640
1    43.149162
Name: delivery_delayed, dtype: float64

=== Promedio de delay_hours por riesgo_proveedor_transportista ===
riesgo_proveedor_transportista
0    20.80
1    45.17
Name: delay_hours, dtype: float64


In [29]:
# Condiciones que superaron el promedio de retraso (25.30%) en el EDA
condiciones_adversas = ['Storm', 'Snow', 'Heavy Rain', 'Fog']

df['clima_adverso'] = df['weather_condition'].isin(condiciones_adversas).astype(int)

# === Validación contra el target ===
print("=== Distribución de clima_adverso ===")
print(df['clima_adverso'].value_counts())
print(f"\n% del total: {df['clima_adverso'].mean()*100:.2f}% de los envios enfrentan clima adverso")

print("\n=== % de retraso por clima_adverso ===")
print(df.groupby('clima_adverso', observed=True)['delivery_delayed'].mean() * 100)

print("\n=== Promedio de delay_hours por clima_adverso ===")
print(df.groupby('clima_adverso', observed=True)['delay_hours'].mean().round(2))

=== Distribución de clima_adverso ===
clima_adverso
0    82025
1    17975
Name: count, dtype: int64

% del total: 17.97% de los envios enfrentan clima adverso

=== % de retraso por clima_adverso ===
clima_adverso
0    23.997562
1    31.232267
Name: delivery_delayed, dtype: float64

=== Promedio de delay_hours por clima_adverso ===
clima_adverso
0    25.21
1    32.38
Name: delay_hours, dtype: float64


In [30]:
# Contar cuantos factores de riesgo coinciden en cada envio
# Factor 1: clima adverso (ya creada)
# Factor 2: trafico severo
# Factor 3: proveedor debil (por debajo del promedio)

trafico_severo = (df['traffic_congestion_level'] == 'Severe').astype(int)
proveedor_debil = (df['supplier_reliability_score'] < df['supplier_reliability_score'].mean()).astype(int)

df['factores_riesgo'] = df['clima_adverso'] + trafico_severo + proveedor_debil

df['nivel_riesgo_integral'] = df['factores_riesgo'].map({
    0: 'Bajo',
    1: 'Medio',
    2: 'Alto',
    3: 'Alto'
})

# === Validacion contra el target ===
print("=== Distribucion de nivel_riesgo_integral ===")
print(df['nivel_riesgo_integral'].value_counts())
print(f"\n% por nivel:")
print(df['nivel_riesgo_integral'].value_counts(normalize=True) * 100)

print("\n=== % de retraso por nivel_riesgo_integral ===")
print(df.groupby('nivel_riesgo_integral', observed=True)['delivery_delayed'].mean().sort_values(ascending=False) * 100)

print("\n=== Promedio de delay_hours por nivel_riesgo_integral ===")
print(df.groupby('nivel_riesgo_integral', observed=True)['delay_hours'].mean().sort_values(ascending=False).round(2))

=== Distribucion de nivel_riesgo_integral ===
nivel_riesgo_integral
Medio    46473
Bajo     29633
Alto     23894
Name: count, dtype: int64

% por nivel:
nivel_riesgo_integral
Medio    46.473
Bajo     29.633
Alto     23.894
Name: proportion, dtype: float64

=== % de retraso por nivel_riesgo_integral ===
nivel_riesgo_integral
Alto     53.758266
Medio    22.763755
Bajo      6.324031
Name: delivery_delayed, dtype: float64

=== Promedio de delay_hours por nivel_riesgo_integral ===
nivel_riesgo_integral
Alto     55.95
Medio    24.06
Bajo      6.56
Name: delay_hours, dtype: float64


In [31]:
# Escalar carrier_rating (rango 2-5) al mismo rango que supplier_reliability_score (rango 43-100)
# Usamos min-max scaling para llevar ambas a escala 0-100

supplier_min = df['supplier_reliability_score'].min()
supplier_max = df['supplier_reliability_score'].max()
carrier_min = df['carrier_rating'].min()
carrier_max = df['carrier_rating'].max()

supplier_escalado = (df['supplier_reliability_score'] - supplier_min) / (supplier_max - supplier_min) * 100
carrier_escalado = (df['carrier_rating'] - carrier_min) / (carrier_max - carrier_min) * 100

df['proveedor_vs_transportista_gap'] = (supplier_escalado - carrier_escalado).round(2)

# === Validacion contra el target ===
print("=== Estadisticas descriptivas del gap ===")
print(df['proveedor_vs_transportista_gap'].describe())

print("\n=== Promedio del gap por delivery_delayed ===")
print(df.groupby('delivery_delayed', observed=True)['proveedor_vs_transportista_gap'].mean().round(2))

print("\n=== Correlacion con delivery_delayed ===")
print(df[['proveedor_vs_transportista_gap', 'delivery_delayed']].corr().round(3))

=== Estadisticas descriptivas del gap ===
count    100000.000000
mean         -8.828596
std          28.199968
min        -100.000000
25%         -28.140000
50%          -8.830000
75%           9.930000
max         100.000000
Name: proveedor_vs_transportista_gap, dtype: float64

=== Promedio del gap por delivery_delayed ===
delivery_delayed
0    -8.22
1   -10.63
Name: proveedor_vs_transportista_gap, dtype: float64

=== Correlacion con delivery_delayed ===
                                proveedor_vs_transportista_gap  \
proveedor_vs_transportista_gap                           1.000   
delivery_delayed                                        -0.037   

                                delivery_delayed  
proveedor_vs_transportista_gap            -0.037  
delivery_delayed                           1.000  


In [32]:
# === Variable 5: almacen_saturado ===
p75_almacen = df['warehouse_utilization'].quantile(0.75)
print(f"Percentil 75 de warehouse_utilization: {p75_almacen}")

df['almacen_saturado'] = (df['warehouse_utilization'] > p75_almacen).astype(int)

print("\n=== Distribucion de almacen_saturado ===")
print(df['almacen_saturado'].value_counts())
print(f"% del total: {df['almacen_saturado'].mean()*100:.2f}%")

print("\n=== % de retraso por almacen_saturado ===")
print(df.groupby('almacen_saturado', observed=True)['delivery_delayed'].mean() * 100)

print("\n=== Correlacion con delivery_delayed ===")
print(df[['almacen_saturado', 'delivery_delayed']].corr().round(3))

# === Variable 7: tiempo_interno_almacen ===
df['tiempo_interno_almacen'] = (df['processing_time_hours'] + df['picking_time_hours'] +
                                 df['packing_time_hours'] + df['loading_time_hours'])

print("\n" + "="*60)
print("\n=== Estadisticas de tiempo_interno_almacen ===")
print(df['tiempo_interno_almacen'].describe().round(2))

print("\n=== Promedio por delivery_delayed ===")
print(df.groupby('delivery_delayed', observed=True)['tiempo_interno_almacen'].mean().round(2))

print("\n=== Correlacion con delivery_delayed ===")
print(df[['tiempo_interno_almacen', 'delivery_delayed']].corr().round(3))

Percentil 75 de warehouse_utilization: 84.7

=== Distribucion de almacen_saturado ===
almacen_saturado
0    75131
1    24869
Name: count, dtype: int64
% del total: 24.87%

=== % de retraso por almacen_saturado ===
almacen_saturado
0    21.912393
1    35.526157
Name: delivery_delayed, dtype: float64

=== Correlacion con delivery_delayed ===
                  almacen_saturado  delivery_delayed
almacen_saturado             1.000             0.135
delivery_delayed             0.135             1.000


=== Estadisticas de tiempo_interno_almacen ===
count    100000.00
mean         50.53
std          20.02
min           7.00
25%          34.00
50%          51.00
75%          68.00
max          94.00
Name: tiempo_interno_almacen, dtype: float64

=== Promedio por delivery_delayed ===
delivery_delayed
0    50.57
1    50.42
Name: tiempo_interno_almacen, dtype: float64

=== Correlacion con delivery_delayed ===
                        tiempo_interno_almacen  delivery_delayed
tiempo_interno_almacen 

### Conclusiones del Feature Engineering

#### Variables creadas (activas)

| Variable | Tipo | Justificación | Validación vs target |
|----------|------|---------------|----------------------|
| `riesgo_proveedor_transportista` | Binaria | Basada en el hallazgo clave Q5: cuando proveedor y transportista están ambos por debajo del promedio, el riesgo se dispara | 43.15% de retraso vs 19.85% del resto. Separa 23.38% de los envíos como alto riesgo |
| `clima_adverso` | Binaria | Agrupa las 4 condiciones climáticas que superaron el promedio de retraso (Storm, Snow, Heavy Rain, Fog) | 31.23% de retraso vs 24.00%. Afecta al 17.97% de los envíos |
| `nivel_riesgo_integral` | Categórica (Bajo/Medio/Alto) | Semáforo que cuenta cuántos de los 3 factores top (clima adverso, tráfico severo, proveedor débil) coinciden en un envío | Traduce los hallazgos del EDA en una variable accionable para operaciones |
| `almacen_saturado` | Binaria | Marca envíos despachados desde almacenes operando por encima del percentil 75 de utilización (>84.7%) | 35.53% de retraso vs 21.91%. Afecta al 24.87% de los envíos |

#### Variables evaluadas y descartadas (con justificación)

| Variable | Razón de descarte |
|----------|-------------------|
| `dias_diferencia_estimado_real` | **Data leakage.** Al calcularla se descubrió que replica exactamente `delay_days` (misma media, std, min y max). Correlación de 0.865 con el target, sospechosamente alta. Usa `actual_delivery_days`, que solo se conoce después de la entrega |
| `proveedor_vs_transportista_gap` | **Sin poder predictivo.** Correlación de -0.037 con el target. El desbalance entre proveedor y transportista no explica retrasos; lo que importa es si ambos son malos simultáneamente (ya capturado en `riesgo_proveedor_transportista`) |
| `tiempo_interno_almacen` | **Sin poder predictivo.** Correlación de -0.003 con el target. El tiempo de procesamiento interno del almacén (picking + packing + loading + processing) no tiene relación con que el envío llegue tarde. El retraso ocurre en el camino, no dentro del almacén |

#### Variables identificadas como Data Leakage (no disponibles al momento de predecir)

Las siguientes variables del dataset original son resultado del mismo evento que queremos predecir
y no estarían disponibles al momento de hacer una predicción en tiempo real. Quedan excluidas
como input para cualquier modelo futuro:

- `delay_days` — cantidad de días de retraso (resultado directo del target)
- `delay_hours` — cantidad de horas de retraso
- `delivery_performance_score` — score calculado después de la entrega
- `delay_reason` — la razón del retraso se conoce solo después de que ocurre
- `actual_delivery_days` — se conoce solo al finalizar la entrega
- `actual_delivery_date` — se conoce solo al finalizar la entrega
- `delivery_attempts` — número de intentos, se acumula durante/después del proceso
- `delivery_successful` — resultado final de la entrega